In [ ]:
import fiftyone as fo
from pathlib import Path
import json
from copy import deepcopy

def get_coco(annotation_file):
    """
    Load a COCO dataset from a JSON annotation file.

    Parameters:
    annotation_file (str): Path to the COCO annotation file.

    Returns:
    dict: Parsed COCO dataset as a dictionary.
    """
    with open(annotation_file, 'r') as f:
        data = json.load(f)
    return data

band = 3 # master of 3,4,7
sensor = 'venus'


dataset_dir = {'venus': "/Data_large/marine/PythonProjects/MMDET/data/interpretation/venus"
            }
labels_path = {'venus': f"/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_{band}.json"
    }


In [ ]:
images_patt = dataset_dir[sensor]

# Ex: your custom label format
# annotations = {
#     "/path/to/images/000001.jpg": [
#         {"bbox": ..., "label": ...},
#         ...
#     ],
#     ...
# }

band = 3 # selects the cocofile type.
cocofile = get_coco(labels_path[sensor])
cocofile


annotations = {}

for image in cocofile['images']:
    image_id = image['id']
    image_filename = image['file_name']
    
    # Get the annotations for the current image
    image_annotations = [ann for ann in cocofile['annotations'] if ann['image_id'] == image_id]
    
    # Convert the annotations to the desired format
    converted_annotations = []
    for ann in image_annotations:
        label_id = ann['category_id']
        label_name = [cat['name'] for cat in cocofile['categories'] if cat['id'] == label_id][0]
        bounding_box = ann['bbox']
        
        converted_annotations.append({
            'label': label_name,
            'bbox': bounding_box
        })
    
    # Add the annotations to the dictionary
    annotations[images_patt + '/' + image_filename.replace('tif','jpg')] = converted_annotations

annotations


In [ ]:
list(annotations.keys())

In [ ]:
from tqdm import tqdm

# Create samples for your data
samples = []

for filepath in tqdm(list(Path(images_patt).rglob("*.jpg"))):
    filepath = filepath.as_posix()
    sample = fo.Sample(filepath=filepath)

    # Convert detections to FiftyOne format
    detections = []
    try:
        for obj in annotations[filepath]:
            label = obj["label"]

            # Bounding box coordinates should be relative values
            # in [0, 1] in the following format:
            # [top-left-x, top-left-y, width, height]
            bounding_box = obj["bbox"]

            detections.append(
                fo.Detection(label=label, bounding_box=bounding_box)
            )

        # Store detections in a field name of your choice
        sample["ground_truth"] = fo.Detections(detections=detections)

        samples.append(sample)
    except KeyError:
        print(f"no annotation for {filepath} in the dataset of testing.")
        continue

# Create dataset
dataset = fo.Dataset("venus-detection-dataset")
dataset.add_samples(samples)

In [ ]:
session = fo.launch_app(dataset)